# LifeMate — lm-A-055: Gemma 4 E2B nghe tiếng Việt chưa train (chỉ đo WER)

**Mở notebook đã quick-save → Save Version → Save & Run All**, T4 x2, Internet On. Input: LifeMate dataset (version 1).

**Vì sao:** bỏ vision thì Gemma 4 E2B còn 4.955B < 5B và nghe được audio trực tiếp. Trước khi tính đến thay Qwen3.5, cần biết nó nghe tiếng Việt tốt đến đâu.

**Làm gì:** chép lời 417 câu cuối dev bằng model gốc, không train, không gọi agent. So trên cùng thước đo: Whisper FT 11.01% và Gipformer FT 10.65% trên 417 câu này. Ước tính ~30 phút.

Output: `gemma_probe/hyp.shard*.json`, `gemma_probe/REPORT.json`.


In [ ]:
import json
MODEL = 'google/gemma-4-E2B-it'
REVISION = '3e22461f65e89153144f8adb70e3b8c2cc9845a7'  # commit 2026-07-20, trước hạn 26/10/2026 của đề
PROMPT = ('Transcribe the following speech segment in Vietnamese into Vietnamese text.\n\n'
          'Follow these specific instructions for formatting the answer:\n'
          '* Only output the transcription, with no newlines.\n'
          '* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three.')
DEV_IDS = ["train_0011", "train_0015", "train_0017", "train_0030", "train_0035", "train_0037", "train_0050", "train_0069", "train_0072", "train_0075", "train_0079", "train_0080", "train_0081", "train_0082", "train_0083", "train_0085", "train_0087", "train_0088", "train_0092", "train_0095", "train_0096", "train_0097", "train_0100", "train_0103", "train_0104", "train_0105", "train_0107", "train_0108", "train_0109", "train_0110", "train_0111", "train_0113", "train_0116", "train_0121", "train_0122", "train_0124", "train_0127", "train_0132", "train_0133", "train_0134", "train_0135", "train_0137", "train_0143", "train_0149", "train_0150", "train_0156", "train_0159", "train_0161", "train_0164", "train_0165", "train_0171", "train_0172", "train_0174", "train_0178", "train_0180", "train_0181", "train_0184", "train_0187", "train_0189", "train_0190", "train_0192", "train_0195", "train_0197", "train_0199", "train_0203", "train_0204", "train_0205", "train_0206", "train_0207", "train_0212", "train_0215", "train_0216", "train_0219", "train_0222", "train_0226", "train_0228", "train_0229", "train_0230", "train_0231", "train_0232", "train_0234", "train_0236", "train_0238", "train_0239", "train_0240", "train_0244", "train_0249", "train_0252", "train_0255", "train_0256", "train_0257", "train_0258", "train_0259", "train_0260", "train_0261", "train_0262", "train_0265", "train_0266", "train_0270", "train_0271", "train_0277", "train_0279", "train_0280", "train_0281", "train_0282", "train_0284", "train_0290", "train_0291", "train_0293", "train_0296", "train_0298", "train_0299", "train_0300", "train_0302", "train_0304", "train_0306", "train_0308", "train_0310", "train_0311", "train_0312", "train_0313", "train_0314", "train_0315", "train_0318", "train_0319", "train_0320", "train_0321", "train_0323", "train_0325", "train_0327", "train_0331", "train_0334", "train_0337", "train_0339", "train_0347", "train_0349", "train_0352", "train_0353", "train_0354", "train_0356", "train_0358", "train_0364", "train_0365", "train_0367", "train_0368", "train_0371", "train_0376", "train_0384", "train_0392", "train_0394", "train_0396", "train_0398", "train_0406", "train_0410", "train_0412", "train_0413", "train_0416", "train_0418", "train_0422", "train_0426", "train_0429", "train_0448", "train_0451", "train_0452", "train_0453", "train_0462", "train_0466", "train_0477", "train_0479", "train_0481", "train_0490", "train_0494", "train_0496", "train_0506", "train_0510", "train_0511", "train_0516", "train_0527", "train_0529", "train_0533", "train_0539", "train_0544", "train_0557", "train_0560", "train_0568", "train_0578", "train_0579", "train_0588", "train_0593", "train_0595", "train_0599", "train_0624", "train_0625", "train_0637", "train_0639", "train_0643", "train_0668", "train_0671", "train_0674", "train_0676", "train_0677", "train_0679", "train_0684", "train_0685", "train_0703", "train_0707", "train_0713", "train_0728", "train_0734", "train_0740", "train_0743", "train_0749", "train_0752", "train_0753", "train_0757", "train_0767", "train_0769", "train_0772", "train_0775", "train_0784", "train_0786", "train_0790", "train_0792", "train_0803", "train_0808", "train_0811", "train_0815", "train_0823", "train_0825", "train_0826", "train_0832", "train_0833", "train_0838", "train_0839", "train_0847", "train_0849", "train_0853", "train_0855", "train_0861", "train_0871", "train_0876", "train_0887", "train_0892", "train_0897", "train_0905", "train_0921", "train_0922", "train_0923", "train_0925", "train_0929", "train_0931", "train_0933", "train_0936", "train_0943", "train_0947", "train_0950", "train_0951", "train_0957", "train_0958", "train_0976", "train_0977", "train_0992", "train_0997", "train_1012", "train_1016", "train_1017", "train_1029", "train_1038", "train_1039", "train_1041", "train_1057", "train_1074", "train_1079", "train_1081", "train_1082", "train_1099", "train_1103", "train_1104", "train_1107", "train_1113", "train_1115", "train_1123", "train_1124", "train_1127", "train_1128", "train_1136", "train_1137", "train_1142", "train_1143", "train_1145", "train_1155", "train_1158", "train_1160", "train_1162", "train_1163", "train_1168", "train_1173", "train_1175", "train_1178", "train_1185", "train_1191", "train_1192", "train_1205", "train_1211", "train_1215", "train_1219", "train_1227", "train_1228", "train_1233", "train_1240", "train_1247", "train_1248", "train_1250", "train_1252", "train_1257", "train_1260", "train_1261", "train_1262", "train_1265", "train_1282", "train_1285", "train_1286", "train_1287", "train_1290", "train_1293", "train_1305", "train_1313", "train_1316", "train_1317", "train_1319", "train_1325", "train_1347", "train_1348", "train_1354", "train_1364", "train_1365", "train_1371", "train_1372", "train_1375", "train_1377", "train_1381", "train_1387", "train_1391", "train_1397", "train_1407", "train_1411", "train_1413", "train_1418", "train_1419", "train_1425", "train_1428", "train_1432", "train_1433", "train_1439", "train_1448", "train_1450", "train_1452", "train_1458", "train_1460", "train_1478", "train_1479", "train_1486", "train_1489", "train_1495", "train_1496", "train_1499", "train_1514", "train_1515", "train_1517", "train_1520", "train_1523", "train_1527", "train_1531", "train_1537", "train_1551", "train_1552", "train_1553", "train_1560", "train_1562", "train_1570", "train_1577", "train_1582", "train_1590", "train_1602", "train_1615", "train_1621", "train_1623", "train_1629", "train_1630", "train_1635", "train_1638", "train_1641", "train_1645", "train_1652", "train_1653", "train_1658", "train_1660", "train_1662", "train_1664", "train_1665", "train_1667", "train_1670", "train_1675", "train_1678", "train_1682", "train_1686", "train_1688", "train_1695", "train_1702", "train_1721", "train_1728", "train_1733", "train_1735", "train_1736", "train_1739", "train_1740", "train_1746"]
print(len(DEV_IDS))


In [ ]:
%%writefile /kaggle/working/asr_fuse.py
"""lm-A-054: word-level fusion of two ASR hypotheses with a count-based language model.

No neural parameters: trigram counts from training text only, plus the words the agent already has for this sample
(profile, assistant turns, tool results). The primary hypothesis keeps its own formatting unless a word really differs.
"""
from __future__ import annotations

import difflib
import math
import re
import unicodedata
from collections import Counter

DIGITS = "không một hai ba bốn năm sáu bảy tám chín".split()
UNITS = [(10**9, "tỷ"), (10**6, "triệu"), (1000, "nghìn")]


def _below_thousand(n: int, full: bool) -> list[str]:
    out = []
    hundreds, rest = n // 100, n % 100
    if hundreds or full:
        out += [DIGITS[hundreds], "trăm"]
    tens, ones = rest // 10, rest % 10
    if tens == 0:
        if ones:
            out += (["linh"] if out else []) + [DIGITS[ones]]
    elif tens == 1:
        out.append("mười")
        if ones:
            out.append("lăm" if ones == 5 else DIGITS[ones])
    else:
        out += [DIGITS[tens], "mươi"]
        if ones:
            out.append({1: "mốt", 4: "tư", 5: "lăm"}.get(ones, DIGITS[ones]))
    return out


def number_words(digits: str) -> list[str]:
    """How a digit string is read aloud. Leading zeros and long strings (phone numbers) are read digit by digit."""
    if (digits.startswith("0") and len(digits) > 1) or len(digits) > 12:
        return [DIGITS[int(d)] for d in digits]
    n = int(digits)
    if n == 0:
        return ["không"]
    out = []
    for value, name in UNITS:
        if n >= value:
            head = n // value
            out += (number_words(str(head)) if head >= 1000 else _below_thousand(head, False)) + [name]
            n %= value
    if n:
        out += _below_thousand(n, bool(out))
    return out


def spoken(token: str) -> list[str]:
    """One raw token as spoken words: lower case, no punctuation, digits read out. '100.000' is one number."""
    token = unicodedata.normalize("NFC", token).lower()
    token = re.sub(r"(?<=\d)[.,](?=\d{3}(?:\D|$))", "", token)
    out = []
    for part in re.findall(r"\w+", token):
        if part.isdigit():
            out += number_words(part)
        else:
            out.append(part)
    return out


def spoken_text(text: str) -> list[str]:
    return [w for token in text.split() for w in spoken(token)]


class TrigramLM:
    def __init__(self, texts):
        self.uni, self.bi, self.tri = Counter(), Counter(), Counter()
        for text in texts:
            w = ["<s>", "<s>"] + spoken_text(text) + ["</s>"]
            self.uni.update(w)
            self.bi.update(zip(w, w[1:]))
            self.tri.update(zip(w, w[1:], w[2:]))
        self.total, self.vocab = sum(self.uni.values()), len(self.uni)

    def logp(self, a: str, b: str, c: str) -> float:
        if self.tri[(a, b, c)]:
            return math.log(self.tri[(a, b, c)] / self.bi[(a, b)])
        if self.bi[(b, c)]:
            return math.log(.4) + math.log(self.bi[(b, c)] / self.uni[b])
        return 2 * math.log(.4) + math.log((self.uni[c] + .5) / (self.total + .5 * self.vocab + .5))

    def score(self, left: list[str], words: list[str], right: list[str]) -> float:
        seq = left[-2:] + words + right[:2]
        return sum(self.logp(seq[i - 2], seq[i - 1], seq[i]) for i in range(2, len(seq)))


def _match_case(word: str, model: str) -> str:
    return word.capitalize() if model[:1].isupper() else word


def fuse(primary: str, secondary: str, lm: TrigramLM, context: set[str], bias: float = -2., context_weight: float = 2.,
         length_penalty: float = 1.) -> str:
    """Primary text with only the disagreeing spans replaced when the language model prefers the secondary words."""
    tokens = primary.split()
    flat, owner = [], []
    for i, token in enumerate(tokens):
        words = spoken(token)
        flat += words
        owner += [i] * len(words)
    other = spoken_text(secondary)
    if not flat or not other:
        return primary
    ops = difflib.SequenceMatcher(None, flat, other, autojunk=False).get_opcodes()
    out, left = [], ["<s>", "<s>"]
    done = 0  # next primary token not yet emitted
    for op, i, j, u, v in ops:
        if op == "equal":
            for t in range(done, owner[j - 1] + 1):
                out.append(tokens[t])
            done = max(done, owner[j - 1] + 1)
            left += flat[i:j]
            continue
        a, b = flat[i:j], other[u:v]
        # A span cutting through one primary token ("118" read as several words) cannot be replaced cleanly.
        inside = 0 < i < len(flat) and owner[i - 1] == owner[i] or (a and j < len(flat) and owner[j - 1] == owner[j])
        right = flat[j:j + 2] + (["</s>"] if j + 2 > len(flat) else [])
        def total(words):
            return lm.score(left, words, right) + context_weight * sum(w in context for w in words) - length_penalty * len(words)
        if inside or total(a) + bias >= total(b):
            if a:
                for t in range(done, owner[j - 1] + 1):
                    out.append(tokens[t])
                done = max(done, owner[j - 1] + 1)
            left += a
            continue
        model = tokens[owner[i]] if a else ""
        if a:
            for t in range(done, owner[i]):
                out.append(tokens[t])
        out += [_match_case(w, model if k == 0 else "") for k, w in enumerate(b)]
        if a:
            done = max(done, owner[j - 1] + 1)
        left += b
    out += tokens[done:]
    return " ".join(out)


In [ ]:
%%writefile /kaggle/working/gemma_probe_worker.py
"""One GPU: transcribe its share of the audio with the base model. float16 first; a T4 has no native bfloat16."""
import json, sys, time
from pathlib import Path
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
shard = int(sys.argv[2])
items = cfg["items"][shard::2]
processor = AutoProcessor.from_pretrained(cfg["model"], revision=cfg["revision"])

def load(dtype):
    return AutoModelForMultimodalLM.from_pretrained(cfg["model"], revision=cfg["revision"], dtype=dtype).to("cuda").eval()

def hear(model, path):
    messages = [{"role": "user", "content": [{"type": "text", "text": cfg["prompt"]}, {"type": "audio", "audio": path}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True).to(model.device)
    inputs = {k: (v.to(model.dtype) if torch.is_floating_point(v) else v) for k, v in inputs.items()}
    n = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

t0 = time.time()
model, dtype = load(torch.float16), "float16"
if not hear(model, items[0]["path"]):  # fp16 can overflow to an empty answer on Gemma
    del model; torch.cuda.empty_cache()
    model, dtype = load(torch.bfloat16), "bfloat16"
load_s = time.time() - t0
hyp, t0 = {}, time.time()
for i, it in enumerate(items):
    hyp[it["id"]] = hear(model, it["path"])
    if i % 25 == 0:
        print(f"[shard {shard}] {i}/{len(items)} {(time.time() - t0) / (i + 1):.2f} s/audio | {hyp[it['id']][:90]}", flush=True)
Path(cfg["out"], f"hyp.shard{shard}.json").write_text(json.dumps({"dtype": dtype, "model_load_s": load_s, "s_per_audio": (time.time() - t0) / len(items),
    "peak_vram_mb": torch.cuda.max_memory_allocated() / 2**20, "hyp": hyp}, ensure_ascii=False), encoding="utf-8")


In [ ]:
import glob, json, os, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working'); OUT = WORK / 'gemma_probe'; OUT.mkdir(exist_ok=True)
assert len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines()) == 2, 'Chọn T4 x2'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'librosa', 'soundfile'], check=True)
train = Path(sorted(glob.glob('/kaggle/input/**/train/train.jsonl', recursive=True))[0])
rows = {r['id']: r for r in map(json.loads, train.read_text(encoding='utf-8').splitlines())}
items = [{'id': i, 'path': str(train.parent / rows[i]['audio'])} for i in DEV_IDS]
(OUT / 'config.json').write_text(json.dumps({'model': MODEL, 'revision': REVISION, 'prompt': PROMPT, 'items': items, 'out': str(OUT)}), encoding='utf-8')
from huggingface_hub import snapshot_download
snapshot_download(MODEL, revision=REVISION)
env = lambda g: dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false')
logs = [open(OUT / f'worker{g}.log', 'w') for g in (0, 1)]
procs = [subprocess.Popen([sys.executable, str(WORK / 'gemma_probe_worker.py'), str(OUT / 'config.json'), str(g)], env=env(g), stdout=logs[g], stderr=subprocess.STDOUT) for g in (0, 1)]
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for g in (0, 1):
        lines = (OUT / f'worker{g}.log').read_text(errors='replace').splitlines()
        if lines: print(lines[-1][:170], flush=True)
for g, p in enumerate(procs):
    print((OUT / f'worker{g}.log').read_text(errors='replace')[-1200:])
    if p.returncode: raise RuntimeError(f'Worker {g} lỗi {p.returncode}')
recs = [json.loads((OUT / f'hyp.shard{g}.json').read_text(encoding='utf-8')) for g in (0, 1)]
hyp = {k: v for r in recs for k, v in r['hyp'].items()}
sys.path.insert(0, str(WORK))
from asr_fuse import spoken_text
def ed(a, b):
    d = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        p, d[0] = d[0], i
        for j, y in enumerate(b, 1):
            p, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, p + (x != y))
    return d[-1]
gold = {i: spoken_text(rows[i]['audio_text']) for i in DEV_IDS}
errs = sum(ed(gold[i], spoken_text(hyp[i])) for i in DEV_IDS); words = sum(map(len, gold.values()))
report = {'wer': errs / words, 'n': len(hyp), 'empty': sum(not t for t in hyp.values()), 'dtype': [r['dtype'] for r in recs],
          's_per_audio': [r['s_per_audio'] for r in recs], 'peak_vram_mb': [r['peak_vram_mb'] for r in recs], 'load_s': [r['model_load_s'] for r in recs],
          'reference': {'whisper_ft': 0.1101, 'gipformer_ft': 0.1065}}
(OUT / 'REPORT.json').write_text(json.dumps(report, indent=1))
print('REPORT', report)
for i in DEV_IDS[:15]:
    print('G', rows[i]['audio_text'][:100], '\nH', hyp[i][:100], '\n')
